In [8]:
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import whisper
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from pydub import AudioSegment
from pydub.silence import split_on_silence

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
print()

Device: CPU



In [9]:
# Load Whisper for English transcription
print("Loading Whisper model...")
whisper_model = whisper.load_model("small")  # Using 'small' for better accuracy
print("✓ Whisper loaded\n")

# Load translation model (NLLB for Telugu/Hindi)
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("✓ Translation model loaded\n")

# Language codes for Telugu and Hindi
LANG_CODES = {
    'telugu': 'tel_Telu',
    'hindi': 'hin_Deva'
}

def audio_to_segments(audio_path):
    audio = AudioSegment.from_file(audio_path)
    
    # Split audio on silence for better transcription
    segments = split_on_silence(
        audio,
        min_silence_len=500,
        silence_thresh=audio.dBFS - 14,
        keep_silence=250
    )
    
    return segments

def transcribe_audio_segment(audio_segment):
    """Transcribe a single audio segment (from project1.ipynb)"""
    import time
    
    # Normalize audio volume
    audio_segment = audio_segment.normalize()
    
    # Export segment to temporary file
    temp_file = f"temp_segment_{time.time()}.wav"
    audio_segment.export(
        temp_file, 
        format="wav",
        parameters=["-ar", "16000", "-ac", "1"]  # 16kHz sample rate, mono
    )
    
    try:
        # Transcribe in English
        result = whisper_model.transcribe(
            temp_file,
            language='en',
            task='transcribe',
            fp16=False
        )
        english_text = result['text'].strip()
        
        if english_text:
            return english_text
        else:
            return None
                    
    except Exception as e:
        print(f"    Transcription error: {str(e)[:100]}")
        return None
                
    finally:
        # Clean up temp file
        try:
            time.sleep(0.1)
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except:
            pass

def translate_text(text, output_language):
    """Translate English text to Telugu or Hindi using NLLB"""
    if not text or not text.strip():
        return ""
    
    target_code = LANG_CODES.get(output_language.lower())
    if not target_code:
        raise ValueError(f"Language must be 'telugu' or 'hindi', got: {output_language}")
    
    tokenizer.src_lang = "eng_Latn"
    
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        generated = translation_model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
            max_length=512,
            num_beams=5,
            early_stopping=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            length_penalty=1.0
        )
    
    translated_text = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    return translated_text

def transcribe_and_translate(audio_file, output_language):
    print(f"Processing audio file: {audio_file}\n")
    
    # Step 1: Split audio into segments (better transcription quality)
    segments = audio_to_segments(audio_file)
    print(f"Found {len(segments)} segments\n")
    
    results = []
    translated_segments = []
    
    # Step 2: Transcribe AND translate each segment individually
    for i, segment in enumerate(segments):
        print(f"Processing segment {i+1}/{len(segments)}...")
        english_text = transcribe_audio_segment(segment)
        
        if not english_text:
            print(f"  No speech detected in segment {i+1}\n")
            continue
        
        print(f"  English: {english_text}")
        
        # Translate this segment immediately (better accuracy)
        translated_text = translate_text(english_text, output_language)
        print(f"  {output_language.capitalize()}: {translated_text}")
        
        results.append({
            'segment': i+1,
            'english_text': english_text,
            'translated_text': translated_text
        })
        translated_segments.append(translated_text)
        print()
    
    # Step 3: Combine all segments
    complete_transcript_english = " ".join([result['english_text'] for result in results])
    complete_transcript_translated = " ".join(translated_segments)
    
    print(f"✓ Processing complete\n")
    
    return {
        'english': complete_transcript_english,
        'translated': complete_transcript_translated,
        'segments': results,
        'language': output_language
    }

print("Ready to process audio!")

Loading Whisper model...
✓ Whisper loaded

Loading translation model...
✓ Translation model loaded

Ready to process audio!


In [10]:
AUDIO_FILE = "audio_files\dog.wav"
OUTPUT_LANGUAGE = "telugu"  # Change to 'telugu' or 'hindi'
result = transcribe_and_translate(AUDIO_FILE, OUTPUT_LANGUAGE)
print("=" * 80)
print("RESULTS")
print("=" * 80)
print(f"\nEnglish Transcript:\n{result['english']}\n")
print(f"\n{OUTPUT_LANGUAGE.capitalize()} Transcript:\n{result['translated']}\n")

Processing audio file: audio_files\dog.wav

Found 54 segments

Processing segment 1/54...
  English: It was a perfect Saturday. The sun was hot and the grass was green. Ten year old Sam stood in the park with his best friend, a golden dog named Rusty.
  Telugu: ఇది ఒక ఖచ్చితమైన శనివారం. సూర్యుడు వేడిగా ఉన్నాడు మరియు గడ్డి ఆకుపచ్చగా ఉంది. 10 ఏళ్ల సామ్ తన బెస్ట్ ఫ్రెండ్, రస్టీ అనే బంగారు కుక్కతో పార్క్ లో నిలబడి ఉన్నాడు.

Processing segment 2/54...
  English: Ready boy.
  Telugu: సిద్ధంగా బాలుడు.

Processing segment 3/54...
  English: Sam lost
  Telugu: సామ్ కోల్పోయింది

Processing segment 4/54...
  English: Rusty wracked his tail so hard his whole body shook.
  Telugu: రస్టీ తన తోక విరిగింది కాబట్టి అతని మొత్తం శరీరం వణుకుతుంది.

Processing segment 5/54...
  English: Sam threw the bright red ball high into the air. Rusty ran fast.
  Telugu: సామ్ ప్రకాశవంతమైన ఎరుపు బంతిని ఎగువకు త్రోసివేసింది. రస్టీ వేగంగా పరుగెత్తాడు.

Processing segment 6/54...
  English: He caught the ball and brought